# Auditoría de la API de abandono bancario

Este notebook verifica el servicio publicado en Render y permite realizar inferencias de ejemplo de forma segura.

In [ ]:
import time
import requests
import pandas as pd

BASE_URL = "https://api-abandono-banca.onrender.com"
TIMEOUT = 30
print(f"API objetivo: {BASE_URL}")

## Comprobación de disponibilidad

In [ ]:
rutas = ["/", "/docs", "/openapi.json", "/health"]
registros = []
for ruta in rutas:
    inicio = time.perf_counter()
    try:
        respuesta = requests.get(BASE_URL + ruta, timeout=TIMEOUT)
        registros.append({"ruta": ruta, "status": respuesta.status_code,
                          "tiempo_s": round(time.perf_counter() - inicio, 3),
                          "tipo": respuesta.headers.get("content-type", ""),
                          "bytes": len(respuesta.content), "resultado": "APROBADO" if respuesta.ok else "FALLIDO"})
    except Exception as exc:
        registros.append({"ruta": ruta, "status": None, "tiempo_s": None,
                          "tipo": None, "bytes": 0, "resultado": f"ERROR: {exc}"})
pd.DataFrame(registros)

## Inferencia de prueba

In [ ]:
cliente = {
    "id_cliente": "CLI-8492", "edad": 42, "ingresos_anuales": 52000,
    "saldo_cuenta": 18500, "nivel_estudios": "Grado", "tipo_cuenta": "Nomina",
    "nivel_satisfaccion": "Medio", "num_productos": 3, "tiene_tarjeta": True,
    "canal_preferido": "App Movil", "antiguedad_meses": 36,
}
respuesta = requests.post(BASE_URL + "/predict", json=cliente, timeout=TIMEOUT)
print("Status:", respuesta.status_code)
resultado = respuesta.json()
resultado

## Validación automática

In [ ]:
pruebas = []
pruebas.append({"comprobacion": "POST /predict responde", "resultado": respuesta.status_code == 200})
pruebas.append({"comprobacion": "Incluye probabilidad", "resultado": isinstance(resultado.get("probabilidad_abandono"), (int, float))})
pruebas.append({"comprobacion": "Probabilidad entre 0 y 1", "resultado": 0 <= resultado.get("probabilidad_abandono", -1) <= 1})
pruebas.append({"comprobacion": "Nivel de riesgo válido", "resultado": resultado.get("nivel_riesgo") in {"Bajo", "Medio", "Alto"}})
pruebas.append({"comprobacion": "Clase binaria válida", "resultado": resultado.get("abandono") in {0, 1}})
resumen = pd.DataFrame(pruebas)
resumen